# Trích xuất đặc trưng và Huấn luyện DINOv2 trên CIFAR-10
Notebook này gọi trực tiếp các hàm đã được đóng gói sẵn trong `src/main.py` và `src/data_loader.py`.

In [ ]:
import sys
sys.path.append('../src')

import torch
import matplotlib.pyplot as plt
import json
from data_loader import get_cifar10_dataloaders, DINO_TRANSFORM
from main import extract_features, train_classifier, evaluate_classifier

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

print('Loading DINOv2 model (ViT-Small)...')
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
model = model.to(device)

In [ ]:
print('Loading CIFAR-10 DataLoaders...')
train_loader, test_loader, classes = get_cifar10_dataloaders(batch_size=64, num_workers=2)

print('Extracting features for training set...')
X_train, y_train = extract_features(model, train_loader, device)

print('Extracting features for testing set...')
X_test, y_test = extract_features(model, test_loader, device)

In [ ]:
print('Bắt đầu quá trình huấn luyện Linear Classifier với PyTorch...')
clf = train_classifier(X_train, y_train, device, model_save_name='model_cifar10.pth')

evaluate_classifier(clf, X_test, y_test, device, dataset_name='CIFAR-10')

In [ ]:
# Vẽ biểu đồ Loss History
with open('loss_history.json', 'r') as f:
    loss_history = json.load(f)

plt.figure(figsize=(8, 5))
plt.plot(loss_history, label='Train Loss', color='blue')
plt.title('Training Loss Curve')
plt.xlabel('Epochs')
plt.ylabel('CrossEntropy Loss')
plt.legend()
plt.grid(True)
plt.show()

### Test thử mô hình với một ảnh ngẫu nhiên
Đoạn code dưới đây sẽ tải lại file `.pth` vừa lưu và đưa ra dự đoán trên 1 ảnh của tập test.

In [ ]:
import torch.nn as nn
from PIL import Image

# 1. Khởi tạo lại mô hình phân loại và nạp trọng số đã lưu
num_classes = len(classes)
clf_test = nn.Linear(384, num_classes).to(device)
clf_test.load_state_dict(torch.load('model_cifar10.pth'))
clf_test.eval()

# 2. Lấy 1 ảnh ngẫu nhiên từ test_loader (hoặc bạn có thể dùng Image.open() với đường dẫn ảnh tự do)
images, labels = next(iter(test_loader))
img_input = images[0].unsqueeze(0).to(device) # Lấy ảnh đầu tiên trong batch
true_class = classes[labels[0].item()]

# 3. Tiến hành dự đoán
with torch.no_grad():
    feature = model(img_input)
    output = clf_test(feature)
    _, predicted_idx = torch.max(output, 1)
    predicted_class = classes[predicted_idx.item()]

print(f"Nhãn thực tế (Ground Truth): {true_class}")
print(f"Nhãn dự đoán của mô hình: {predicted_class}")